# CP4 — Task 1: Protokol dan Run Eksperimen (W11)

Notebook ini memenuhi poin wajib W11 nomor 1: **menjalankan experiment track secara penuh pada data validation**.

Yang dilakukan:

1. Memakai protokol CP3 tanpa perubahan: official temporal split, warm-start evaluable users, candidate universe resmi, relevan rating >= 4, item training-seen dikeluarkan.
2. Menjalankan **20 konfigurasi** pada validation: 3 baseline, 11 bobot Weighted Score (alpha 0,0–1,0), dan 6 konstanta RRF (k = 10–100).
3. Menghitung metrik untuk K = 10, 20, 50 (Recall, Precision, NDCG, MRR, MAP).
4. Memilih model terbaik per keluarga dan keseluruhan berdasarkan NDCG@10, dengan Recall@10 sebagai tie-breaker.
5. Mencatat seluruh setelan (seed, alpha, k, cache, commit, hash data) agar reproducible.

**Test set tidak dipakai sama sekali.** Seluruh angka adalah hasil pengembangan di validation.

Notebook ini hanya memanggil fungsi dari `src/w11_experiment.py`; tidak ada rumus metrik atau fusi yang ditulis ulang di sini.

## 1. Setup dan import

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "w11_experiment.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Jalankan notebook dari folder proyek yang berisi src/w11_experiment.py.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.w11_experiment import ALPHAS, K_LIST, PRIMARY_K, RRF_KS, run

pd.set_option("display.width", 180)
print("Project root :", PROJECT_ROOT)
print("Alpha grid   :", ALPHAS)
print("RRF k grid   :", RRF_KS)
print("Metrik K     :", K_LIST, "| metrik utama: NDCG@%d" % PRIMARY_K)

Project root : d:\Semester 7\Sisrek\Praktikum\proyek
Alpha grid   : (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0)
RRF k grid   : (10, 20, 40, 60, 80, 100)
Metrik K     : (10, 20, 50) | metrik utama: NDCG@10


## 2. Menjalankan eksperimen penuh di validation

Sel di bawah menjalankan seluruh pipeline: memuat data dan cache ranking, mengaudit protokol, mengevaluasi 20 konfigurasi, lalu menulis artefak ke `outputs/w11/`.

In [2]:
result = run(PROJECT_ROOT, PROJECT_ROOT / "outputs" / "w11")
print(f"Durasi run : {result['duration_sec']:.1f} detik")
print(f"Konfigurasi: {len(result['log'])}")

audit = pd.DataFrame(result["metadata"]["audit"])
display(audit)

Durasi run : 62.8 detik
Konfigurasi: 20


,pemeriksaan,status,bukti
0,protokol ranking - Popularity,PASS,"11659 user tersedia, 11659 dievaluasi; max 200..."
1,protokol ranking - Item-CF,PASS,"11659 user tersedia, 11659 dievaluasi; max 200..."
2,protokol ranking - TF-IDF CBF,PASS,"11659 user tersedia, 11659 dievaluasi; max 200..."
3,target relevan tidak training-seen,PASS,0 overlap
4,user universe sama untuk semua model,PASS,11659 user per model


## 3. Hasil seluruh konfigurasi

Tabel berikut adalah log eksperimen lengkap, diurutkan menurut NDCG@10. Inilah bukti bahwa eksperimen tidak berhenti pada satu konfigurasi.

In [3]:
log = result["log"].sort_values(["ndcg@10", "recall@10"], ascending=False)
columns = [
    "config_id", "family", "ndcg@10", "recall@10", "ndcg@20", "ndcg@50", "mrr@10", "duration_sec",
]
display(log[columns].round(6).reset_index(drop=True))

,config_id,family,ndcg@10,recall@10,ndcg@20,ndcg@50,mrr@10,duration_sec
0,E1-popularity,baseline,0.009222,0.013499,0.011476,0.016235,0.012988,0.312
1,E5-rrf-k100,rrf,0.009086,0.014198,0.011097,0.014563,0.010687,0.489
2,E5-rrf-k60,rrf,0.008750,0.013565,0.010591,0.014102,0.010173,0.670
3,E5-rrf-k80,rrf,0.008661,0.013259,0.010720,0.014210,0.010226,0.417
4,E5-rrf-k40,rrf,0.008291,0.012633,0.010337,0.013570,0.009749,0.429
5,E4-weighted-a030,weighted,0.007930,0.012820,0.009973,0.013443,0.008818,0.559
6,E5-rrf-k20,rrf,0.007841,0.012407,0.009665,0.012995,0.008951,0.496
7,E4-weighted-a020,weighted,0.007774,0.012455,0.009893,0.013481,0.008627,0.609
8,E4-weighted-a040,weighted,0.007726,0.012401,0.009900,0.013254,0.008650,0.577
9,E4-weighted-a010,weighted,0.007703,0.012670,0.009676,0.013283,0.008386,0.561


## 4. Model terbaik di validation

Pemilihan mengikuti CP3: NDCG@10 tertinggi, lalu Recall@10 sebagai tie-breaker. Baseline ikut dipertimbangkan, sehingga model fusi harus mengalahkan baseline untuk bisa menang.

In [4]:
winners = result["winners"]

rows = [
    {
        "keluarga": family,
        "config_id": row["config_id"],
        "ndcg@10": row["ndcg@10"],
        "recall@10": row["recall@10"],
    }
    for family, row in winners["best_by_family"].items()
]
best = winners["best_overall"]
rows.append(
    {
        "keluarga": "KESELURUHAN",
        "config_id": best["config_id"],
        "ndcg@10": best["ndcg@10"],
        "recall@10": best["recall@10"],
    }
)
display(pd.DataFrame(rows).round(6))

print(f"Model terbaik keseluruhan: {best['config_id']}")
print(f"NDCG@{PRIMARY_K} = {best['ndcg@10']:.6f} | Recall@{PRIMARY_K} = {best['recall@10']:.6f}")

,keluarga,config_id,ndcg@10,recall@10
0,baseline,E1-popularity,0.009222,0.013499
1,rrf,E5-rrf-k100,0.009086,0.014198
2,weighted,E4-weighted-a030,0.007930,0.012820
3,KESELURUHAN,E1-popularity,0.009222,0.013499


Model terbaik keseluruhan: E1-popularity
NDCG@10 = 0.009222 | Recall@10 = 0.013499


## 5. Reproducibility

Setelan yang dicatat: seed, grid hyperparameter, cache key, commit Git, hash file data, dan status pemakaian test. Semuanya juga tersimpan di `outputs/w11/w11_metadata.json`.

In [5]:
meta = result["metadata"]
protocol = meta["protocol"]

print("Seed                 :", meta["seed"])
print("Konfigurasi          :", meta["n_configs"])
print("Alpha grid           :", meta["search_space"]["alphas"])
print("RRF k grid           :", meta["search_space"]["rrf_ks"])
print("Cache key            :", meta["cache"]["cache_key"])
print("Commit               :", meta["git"]["commit"][:12], "| dirty:", meta["git"]["dirty"])
print("Train interactions   :", f"{protocol['train_interactions']:,}")
print("Validation warm      :", f"{protocol['validation_warm_rows']:,} baris")
print("Evaluable users      :", f"{protocol['evaluable_users']:,}")
print("Candidate universe   :", f"{protocol['candidate_universe']:,}")
print("Target relevan       :", f"{protocol['relevant_targets']:,}")
print("Test dipakai         :", protocol["uses_final_test"])
print("Artefak              :", (PROJECT_ROOT / "outputs" / "w11").relative_to(PROJECT_ROOT))

Seed                 : 42
Konfigurasi          : 20
Alpha grid           : [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
RRF k grid           : [10, 20, 40, 60, 80, 100]
Cache key            : 13454c5090f84bab
Commit               : e7ac08753054 | dirty: True
Train interactions   : 427,957
Validation warm      : 27,537 baris
Evaluable users      : 11,659
Candidate universe   : 22,753
Target relevan       : 24,202
Test dipakai         : False
Artefak              : outputs\w11


## 6. Kesimpulan Task 1

- Eksperimen track dijalankan penuh di validation: 20 konfigurasi pada 11.659 user dengan candidate universe resmi 22.753 item.
- Hyperparameter search sesuai search space CP3: 11 nilai alpha dan 6 nilai k RRF, bukan satu konfigurasi.
- Model terbaik sudah teridentifikasi berdasarkan NDCG@10 di validation.
- Log lengkap tersimpan di `outputs/w11/w11_all_configs.csv`, metadata dan audit di `outputs/w11/w11_metadata.json`.
- Test set tidak disentuh; seluruh angka adalah hasil pengembangan dan baru dievaluasi final di W13.